# IZQAR — BAF Base account-opening fraud
Add Input: `sgpjesus/bank-account-fraud-dataset-neurips-2022` and choose only `Base.csv`. This is a separate application-fraud task, not a transfer or payment model. Training uses earlier months, validation the penultimate month, and one-time final test the last month. The target `fraud_bool`, `month`, post-request fields and unverified history features are excluded from X. The data is synthetic; metrics are not bank performance.

In [ ]:
from pathlib import Path
import subprocess, sys, json
EXPECTED_REVISION = '5111638e3e3a4ae1c8aee948525de0a95fd5d679'
root = Path('/kaggle/working/izqar_baf')
if root.exists():
    raise FileExistsError('Use a fresh session; refusing to reuse a stale code checkout')
subprocess.run(['git', 'clone', '--branch', 'feat/ml-pipelines-and-startup',
                'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '-r', str(backend / 'requirements-ml.txt'), 'scikit-learn==1.9.1'], check=True)
candidates = [p for p in Path('/kaggle/input').rglob('Base.csv')
              if 'bank-account-fraud' in str(p).lower()]
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected one BAF Base.csv via Add Input; found {len(candidates)}')
data = candidates[0]
print('Input:', data, 'bytes:', data.stat().st_size)
actual_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
if actual_revision != EXPECTED_REVISION:
    raise RuntimeError('Required reviewed revision was not checked out; do not train stale code.')
print('Code commit:', actual_revision)

In [ ]:
validation_report = Path('/kaggle/working/baf_base_validation_v1.metrics.json')
subprocess.run([sys.executable, str(backend / 'scripts/train_baf.py'),
                '--data', str(data), '--output', '/kaggle/working/baf_base_validation_v1.joblib',
                '--mode', 'experiment', '--estimators', '180',
                '--max-false-alerts-per-10000', '100'], cwd=backend, check=True)
assert json.loads(validation_report.read_text())['test_metrics'] is None
print('Review validation metrics and threshold budget before finalizing:', validation_report)

In [ ]:
RUN_FINAL = False  # Set True only after reviewing validation and fixing all settings.
if RUN_FINAL:
    final_model = Path('/kaggle/working/baf_base_final_v1.joblib')
    subprocess.run([sys.executable, str(backend / 'scripts/train_baf.py'),
                    '--data', str(data), '--output', str(final_model),
                    '--mode', 'final', '--estimators', '180',
                    '--max-false-alerts-per-10000', '100'], cwd=backend, check=True)
    final_report = json.loads(final_model.with_suffix('.metrics.json').read_text())
    assert final_report['model_kind'] == 'baf_base' and final_report['test_metrics'] is not None
    portable = Path('/kaggle/working/izqar_baf_base_v1_portable.zip')
    subprocess.run([sys.executable, str(backend / 'scripts/export_portable_model.py'),
                    '--source', str(final_model), '--output-stem', '/kaggle/working/baf_base_v1_portable',
                    '--bundle', str(portable)], cwd=backend, check=True)
    print('Download trusted model bundle:', portable)
else:
    print('Final test remains sealed. Review validation report first.')